In [7]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import iter_transaction_chunks, select_period

csv_path = project_root / "data/raw/archive/HI-Small_Trans.csv"

endpoint_columns = [
    "sender_bank",
    "sender_account",
    "receiver_bank",
    "receiver_account",
]

In [8]:
history_parts = []
row_offset = 0

chunks = iter_transaction_chunks(
    csv_path,
    usecols=["timestamp", *endpoint_columns],
)

for chunk in chunks:
    # Assign source positions before filtering, exactly as in Lab 2.
    chunk["source_row"] = range(
        row_offset,
        row_offset + len(chunk),
    )
    row_offset += len(chunk)

    chunk["timestamp"] = pd.to_datetime(
        chunk["timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise",
    )

    selected = select_period(
        chunk,
        start=pd.Timestamp("2022-09-01"),
        end=pd.Timestamp("2022-09-09"),
    )

    if not selected.empty:
        for column in endpoint_columns:
            selected[column] = selected[column].astype(
                "string[pyarrow]"
            )

        history_parts.append(selected)

history = (
    pd.concat(history_parts, ignore_index=True)
    .sort_values(["timestamp", "source_row"])
    .reset_index(drop=True)
)

# Release the intermediate list after combining it.
history_parts.clear()

print(f"History transactions: {len(history):,}")
print(
    "History memory:",
    round(history.memory_usage(deep=True).sum() / 1024**2, 1),
    "MiB",
)
display(history.head())

History transactions: 4,214,445
History memory: 313.6 MiB


,timestamp,sender_bank,sender_account,receiver_bank,receiver_account,source_row
0,2022-09-01,03209,8000F4670,03209,8000F4670,2
1,2022-09-01,01420,8005DFEB0,01420,8005DFEB0,17
2,2022-09-01,010,8000F6850,010,8000F6850,158
3,2022-09-01,012,8001167D0,012,8001167D0,218
4,2022-09-01,001,8001364A0,001,8001364A0,281


In [10]:
import importlib
import src.data

importlib.reload(src.data)

<module 'src.data' from '/Users/sara/Projects/data-science-lab/src/data.py'>

In [11]:
from src.data import attach_account_endpoints

train_base = pd.read_parquet(
    project_root / "data/processed/lab02_train.parquet"
)
validation_base = pd.read_parquet(
    project_root / "data/processed/lab02_validation.parquet"
)

train = attach_account_endpoints(train_base, history)
validation = attach_account_endpoints(validation_base, history)

assert len(history) == 4_214_445
assert history["source_row"].is_unique

assert len(train) == 1_379_551
assert len(validation) == 965_524

assert int(train["is_laundering"].sum()) == 1_800
assert int(validation["is_laundering"].sum()) == 1_036

display(
    train[
        ["source_row", "timestamp", *endpoint_columns]
    ].head()
)

,source_row,timestamp,sender_bank,sender_account,receiver_bank,receiver_account
0,1870235,2022-09-03,01292,80015F700,010,80015F8E0
1,1870247,2022-09-03,012,8001A8CC0,0111425,80B7FE8F0
2,1870250,2022-09-03,070,100428660,01362,8001C4B70
3,1870255,2022-09-03,012,8001CB7C0,0011642,804DE86F0
4,1870258,2022-09-03,001,8001C9040,010,8001CC420


### Historical-feature setup

We loaded account endpoints and timestamps for September 1–8.
September 1–2 supplies earlier context for training transactions.

We restored endpoints using source-row identifiers, checking one-to-one
matching, timestamp agreement, and transaction order.

Historical features will use the preceding 24 hours, excluding the
current timestamp. Earlier validation activity is available to later
transactions, but laundering labels are excluded from history features.

In [13]:
from time import perf_counter

from src.history import HISTORY_FEATURES, make_history_features

# Targets are the transactions we will train on or evaluate.
# History also includes September 1–2 as earlier context.
targets = select_period(
    history,
    start=pd.Timestamp("2022-09-03"),
    end=pd.Timestamp("2022-09-09"),
)

started = perf_counter()

history_features = make_history_features(history, targets)

print(f"Calculation took {perf_counter() - started:.1f} seconds.")
print(f"Feature rows: {len(history_features):,}")
display(history_features.head())

Calculation took 13.8 seconds.
Feature rows: 2,345,075


,source_row,sender_in_count_24h,receiver_in_count_24h,sender_out_count_24h,receiver_out_count_24h
0,1870235,3,3,9,8
1,1870247,0,2,2,6
2,1870250,539,5,26365,11
3,1870255,1,6,32,0
4,1870258,4,5,23,9


In [14]:
train_with_history = train.merge(
    history_features,
    on="source_row",
    how="left",
    sort=False,
    validate="one_to_one",
)

validation_with_history = validation.merge(
    history_features,
    on="source_row",
    how="left",
    sort=False,
    validate="one_to_one",
)

for original, enriched in [
    (train, train_with_history),
    (validation, validation_with_history),
]:
    assert enriched["source_row"].equals(original["source_row"])
    assert enriched[HISTORY_FEATURES].notna().all().all()
    assert enriched[HISTORY_FEATURES].ge(0).all().all()

display(
    train_with_history[HISTORY_FEATURES]
    .describe(percentiles=[0.5, 0.9, 0.99])
    .round(1)
)

,sender_in_count_24h,receiver_in_count_24h,sender_out_count_24h,receiver_out_count_24h
count,1379551.0,1379551.0,1379551.0,1379551.0
mean,3.6,2.5,826.4,1.5
std,24.6,3.3,3267.6,2.9
min,0.0,0.0,0.0,0.0
50%,1.0,2.0,4.0,0.0
90%,4.0,5.0,429.0,5.0
99%,30.0,11.0,18705.0,13.0
max,539.0,80.0,26365.0,54.0


In [15]:
example = train_with_history.loc[
    train_with_history["sender_in_count_24h"] > 0
].iloc[0]

window_start = example["timestamp"] - pd.Timedelta("24h")

recent = history.loc[
    (history["timestamp"] >= window_start)
    & (history["timestamp"] < example["timestamp"])
]

sender_incoming = recent.loc[
    (recent["receiver_bank"] == example["sender_bank"])
    & (recent["receiver_account"] == example["sender_account"])
]

print("Transaction time:", example["timestamp"])
print("Manually counted:", len(sender_incoming))
print("Feature value:", example["sender_in_count_24h"])

assert len(sender_incoming) == example["sender_in_count_24h"]

display(sender_incoming.head(10))

Transaction time: 2022-09-03 00:00:00
Manually counted: 3
Feature value: 3


,timestamp,sender_bank,sender_account,receiver_bank,receiver_account,source_row
1427497,2022-09-02 09:38:00,010,800126360,01292,80015F700,1424015
1433488,2022-09-02 09:50:00,010,800126360,01292,80015F700,1424014
1664809,2022-09-02 17:20:00,03283,80015F7F0,01292,80015F700,1655098


In [16]:
history_features.to_parquet(
    project_root / "data/processed/lab03_history_features.parquet",
    index=False,
)

In [24]:
import importlib
import numpy as np
import src.modeling

from time import perf_counter
from threadpoolctl import threadpool_limits
from src.history import HISTORY_FEATURES

importlib.reload(src.modeling)

from src.modeling import make_features, build_model, evaluate_scores

history_features = pd.read_parquet(
    project_root / "data/processed/lab03_history_features.parquet"
)

datasets = {}

for split_name in ["train", "validation"]:
    base = pd.read_parquet(
        project_root / f"data/processed/lab02_{split_name}.parquet"
    )

    enriched = base.merge(
        history_features,
        on="source_row",
        how="left",
        sort=False,
        validate="one_to_one",
    )

    assert enriched["source_row"].equals(base["source_row"])
    assert enriched[HISTORY_FEATURES].notna().all().all()

    datasets[split_name] = enriched

train = datasets["train"]
validation = datasets["validation"]

In [25]:
experiments = {
    "transaction_only": (),
    "with_history": tuple(HISTORY_FEATURES),
}

results = {}
models = {}

for experiment_name, extra_features in experiments.items():
    X_train = make_features(
        train,
        extra_numeric_features=extra_features,
    )
    X_validation = make_features(
        validation,
        extra_numeric_features=extra_features,
    )

    model = build_model(
        extra_numeric_features=extra_features,
    )

    started = perf_counter()

    with threadpool_limits(limits=2):
        model.fit(X_train, train["is_laundering"])
        scores = model.predict_proba(X_validation)[:, 1]

    fit_predict_seconds = perf_counter() - started

    metrics = evaluate_scores(
        validation,
        scores,
        daily_budget=50,
    )
    metrics["fit_predict_seconds"] = fit_predict_seconds

    results[experiment_name] = metrics
    models[experiment_name] = model

comparison = pd.DataFrame.from_dict(results, orient="index")

display(comparison.round(4))

,average_precision,alerts,laundering_caught,precision_at_budget,recall_at_budget,fit_predict_seconds
transaction_only,0.0669,100,27,0.27,0.0261,3.7784
with_history,0.1338,100,52,0.52,0.0502,4.0639


In [30]:
saved_baseline = pd.read_csv(
    project_root / "reports/lab_02_baseline_metrics.csv",
    index_col=0,
).loc["gradient_boosting"]

metric_columns = [
    "average_precision",
    "alerts",
    "laundering_caught",
    "precision_at_budget",
    "recall_at_budget",
]

assert np.allclose(
    comparison.loc["transaction_only", metric_columns].to_numpy(
        dtype=float
    ),
    saved_baseline[metric_columns].to_numpy(dtype=float),
)

In [31]:
comparison.to_csv(
    project_root / "reports/lab_03_history_metrics.csv"
)

Adding account history approximately doubled average precision, from 0.0669 to 0.1338. At the same 50-reviews-per-day budget, detections increased from 27 to 52, raising precision from 27% to 52%. Both measures favor the model with history, supporting better overall ranking and better prioritization within our review capacity.